# LeetCode 1562 — Find Latest Group of Size M bằng Python

Link đề: https://leetcode.com/problems/find-latest-group-of-size-m/ · Độ khó: **Medium** · Chủ đề: Array, Hash Table, Binary Search, Simulation

Skeleton LeetCode:

```python
class Solution:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        ...
```

## 1. Mục tiêu học

1. Viết baseline **mô phỏng** từng bước: dựng chuỗi nhị phân, đếm độ dài các nhóm `1`, kiểm tra có nhóm dài đúng `m`.
2. Nắm kỹ thuật **"lưu độ dài ở hai đầu đoạn"**: khi bật một bit, ghép đoạn trái + đoạn phải trong `O(1)` và chỉ cập nhật hai đầu mút.
3. Duy trì **bộ đếm số nhóm theo độ dài** `cnt[len]` để trả lời "hiện có nhóm dài `m` không?" trong `O(1)`.
4. Cài đặt **Union-Find có kích thước** cho cùng bài toán, và một cách nhìn khác: **mảng thời điểm + cửa sổ trượt** `O(n)`.
5. Tránh các bẫy: quên giảm bộ đếm của các nhóm bị gộp, chỉ cập nhật độ dài tại vị trí vừa bật, lệch chỉ số bước.

## 2. Đề bài & ví dụ

Cho mảng `arr` là một **hoán vị** của `1..n`. Có một chuỗi nhị phân độ dài `n`, ban đầu toàn `0`. Ở bước `i` (đánh số từ 1 tới `n`, chuỗi cũng đánh số từ 1), bit tại vị trí `arr[i]` được bật thành `1`.

Cho số nguyên `m`. Tìm **bước muộn nhất** mà tại đó tồn tại một **nhóm `1` có độ dài đúng `m`**. Một nhóm `1` là một đoạn con liên tiếp toàn `1` **không thể mở rộng** về hai phía. Nếu không bao giờ có nhóm như vậy, trả `-1`.

**Ví dụ 1:** `arr = [3,5,1,2,4]`, `m = 1` → `4`.
```
Bước 1: "00100" nhóm ["1"]
Bước 2: "00101" nhóm ["1","1"]
Bước 3: "10101" nhóm ["1","1","1"]
Bước 4: "11101" nhóm ["111","1"]
Bước 5: "11111" nhóm ["11111"]
```
Bước muộn nhất có nhóm độ dài 1 là bước 4.

**Ví dụ 2:** `arr = [3,1,5,4,2]`, `m = 2` → `-1` (không bước nào có nhóm dài đúng 2).

**Ràng buộc:** `n == arr.length`, `1 ≤ m ≤ n ≤ 10⁵`, `1 ≤ arr[i] ≤ n`, các `arr[i]` đôi một khác nhau.

In [1]:
import random
import time
from collections import deque, Counter
from typing import List

examples = [([3, 5, 1, 2, 4], 1, 4), ([3, 1, 5, 4, 2], 2, -1)]
for arr, m, want in examples:
    print(f"arr = {arr}, m = {m} -> mong đợi {want}")

arr = [3, 5, 1, 2, 4], m = 1 -> mong đợi 4
arr = [3, 1, 5, 4, 2], m = 2 -> mong đợi -1


## 3. Kiến thức nền

### 3.1 Tách nhóm `1` trong chuỗi

`s.split("0")` tách theo ký tự `0`; lọc chuỗi rỗng là ra các nhóm `1`. Độ dài mỗi nhóm là `len`.

In [2]:
s = "11101"
groups = [g for g in s.split("0") if g]
print(groups, [len(g) for g in groups])

['111', '1'] [3, 1]


### 3.2 Mảng có "lính canh" ở hai đầu

Dùng mảng độ dài `n + 2` (chỉ số `0` và `n + 1` luôn là `0`) để khỏi kiểm tra biên khi nhìn `x − 1`, `x + 1`.

In [3]:
n = 5
length = [0] * (n + 2)
x = 1
print("nhìn trái của 1:", length[x - 1], "| nhìn phải của 5:", length[5 + 1])

nhìn trái của 1: 0 | nhìn phải của 5: 0


### 3.3 Đếm theo khoá với mảng hoặc `Counter`

`cnt[L]` = số nhóm hiện có độ dài `L`. Khi hai nhóm độ dài `a`, `b` hợp với bit mới thành nhóm `a + b + 1`: `cnt[a] -= 1`, `cnt[b] -= 1`, `cnt[a+b+1] += 1`.

In [4]:
cnt = Counter({1: 3})          # "10101": ba nhóm độ dài 1
a, b = 1, 1                     # bật bit giữa hai nhóm 1
cnt[a] -= 1; cnt[b] -= 1; cnt[a + b + 1] += 1
print(+cnt)                     # dấu + bỏ các khoá có giá trị 0

Counter({1: 1, 3: 1})


### 3.4 Union-Find có kích thước

Mỗi tập lưu `size` ở gốc. `union(a, b)` cộng kích thước. Dùng để biết độ dài nhóm chứa một vị trí.

In [5]:
parent = list(range(6)); size = [1] * 6
def find(x):
    while parent[x] != x:
        parent[x] = parent[parent[x]]
        x = parent[x]
    return x
def union(a, b):
    ra, rb = find(a), find(b)
    if ra != rb:
        parent[ra] = rb
        size[rb] += size[ra]
union(1, 2); union(2, 3)
print("kích thước nhóm chứa 1:", size[find(1)])

kích thước nhóm chứa 1: 3


### 3.5 Cửa sổ trượt lấy max bằng `deque` đơn điệu

Giữ chỉ số trong `deque` sao cho giá trị giảm dần; đầu `deque` là max của cửa sổ. Mỗi chỉ số vào/ra đúng một lần ⇒ `O(n)`.

In [6]:
def window_max(a, k):
    dq, out = deque(), []
    for i, v in enumerate(a):
        while dq and a[dq[-1]] <= v:
            dq.pop()
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()
        if i >= k - 1:
            out.append(a[dq[0]])
    return out

print(window_max([3, 1, 4, 1, 5, 9, 2, 6], 3))

[4, 4, 5, 9, 9, 9]


## 4. Ý tưởng nền tảng / trực giác

Theo dõi ví dụ 1 (`m = 1`) và chú ý **điều gì xảy ra với các nhóm khi bật một bit**:

| Bước | Bật | Trái có nhóm dài | Phải có nhóm dài | Nhóm mới | Các nhóm hiện có | Có nhóm dài 1? |
|------|-----|------------------|------------------|----------|------------------|----------------|
| 1 | 3 | 0 | 0 | 1 | {1} | có |
| 2 | 5 | 0 | 0 | 1 | {1, 1} | có |
| 3 | 1 | 0 | 0 | 1 | {1, 1, 1} | có |
| 4 | 2 | 1 (vị trí 1) | 1 (vị trí 3) | 3 | {3, 1} | có (vị trí 5) |
| 5 | 4 | 3 | 1 | 5 | {5} | **không** |

Bật bit `x` chỉ ảnh hưởng tới **nhóm sát bên trái** và **nhóm sát bên phải** của `x`: hai nhóm đó (nếu có) hợp với `x` thành một nhóm mới dài `left + right + 1`. Mọi nhóm khác giữ nguyên. Vậy chỉ cần biết nhanh **độ dài nhóm kết thúc tại `x − 1`** và **nhóm bắt đầu tại `x + 1`** — đó là lý do lưu độ dài ở **hai đầu mút** của mỗi nhóm.

In [7]:
def show_steps(arr):
    n = len(arr)
    bits = ["0"] * n
    for step, x in enumerate(arr, 1):
        bits[x - 1] = "1"
        s = "".join(bits)
        print(f"bước {step}: bật {x} -> {s}, độ dài nhóm = {[len(g) for g in s.split('0') if g]}")

show_steps([3, 5, 1, 2, 4])

bước 1: bật 3 -> 00100, độ dài nhóm = [1]
bước 2: bật 5 -> 00101, độ dài nhóm = [1, 1]
bước 3: bật 1 -> 10101, độ dài nhóm = [1, 1, 1]
bước 4: bật 2 -> 11101, độ dài nhóm = [3, 1]
bước 5: bật 4 -> 11111, độ dài nhóm = [5]


## 5. Cách 1 — Brute Force / baseline: mô phỏng và đếm lại từ đầu

**Ý tưởng.** Sau mỗi bước, duyệt cả chuỗi để tính độ dài các nhóm; nếu có nhóm dài `m` thì cập nhật `ans = step`. Đáp án là `ans` cuối cùng (hoặc `-1`).

**Độ phức tạp:** `n` bước × `O(n)` mỗi bước = `O(n²)` — với `n = 10⁵` là `10¹⁰` phép, quá chậm.

In [8]:
def brute_simulate(arr, m):
    n = len(arr)
    bits = [0] * n
    ans = -1
    for step, x in enumerate(arr, 1):
        bits[x - 1] = 1
        run = 0
        found = False
        for b in bits + [0]:            # thêm 0 lính canh để chốt nhóm cuối
            if b:
                run += 1
            else:
                if run == m:
                    found = True
                    break
                run = 0
        if found:
            ans = step
    return ans

for arr, m, want in examples:
    print(brute_simulate(arr, m), "mong đợi", want)

4 mong đợi 4
-1 mong đợi -1


In [9]:
# Nút thắt: thời gian tăng bậc hai
rng = random.Random(0)
for n in [500, 1000, 2000]:
    arr = list(range(1, n + 1))
    rng.shuffle(arr)
    t0 = time.perf_counter()
    brute_simulate(arr, n + 1)           # m không bao giờ đạt -> phải duyệt hết mọi bước
    print(f"n={n}: {time.perf_counter() - t0:.3f}s")

n=500: 0.008s
n=1000: 0.031s


n=2000: 0.114s


**Nút thắt:** mỗi bước ta **đếm lại toàn bộ** các nhóm, trong khi mỗi bước chỉ làm thay đổi tối đa 3 nhóm (2 nhóm bị gộp, 1 nhóm mới).

## 6. Ý tưởng thuật toán / bước đột phá

**Insight 1 — chỉ hai đầu mút quan trọng.** Lưu `length[p]` = độ dài nhóm chứa `p`, nhưng **chỉ cần đúng ở hai đầu mút** của nhóm. Khi bật `x`:
- `left = length[x − 1]` (nếu `x − 1` là `1` thì nó là **đầu phải** của nhóm trái — vì `x` đang là `0`),
- `right = length[x + 1]` (đầu trái của nhóm phải),
- nhóm mới dài `total = left + right + 1`, có đầu trái `x − left` và đầu phải `x + right` ⇒ gán `length[x − left] = length[x + right] = total`.

**Insight 2 — đếm nhóm theo độ dài.** `cnt[L]` = số nhóm hiện có độ dài `L`. Mỗi bước: `cnt[left] −= 1` (nếu `left > 0`), `cnt[right] −= 1` (nếu `right > 0`), `cnt[total] += 1`. Sau bước `i`, *nếu* `cnt[m] > 0` *thì* `ans = i`.

**Invariant:** sau mỗi bước, với mọi nhóm `[l, r]`: `length[l] = length[r] = r − l + 1`, và `cnt[L]` đúng bằng số nhóm có độ dài `L`. Các ô bên trong nhóm có thể chứa giá trị cũ — không sao, vì ta không bao giờ đọc chúng nữa (bit mới bật luôn nằm **ngoài** mọi nhóm, nên `x − 1`, `x + 1` nếu là `1` thì chắc chắn là đầu mút).

**Mẹo không cần `cnt`:** trước khi gộp, *nếu* `left == m` hoặc `right == m` *thì* một nhóm dài `m` vừa tồn tại ở bước `i − 1` và sắp biến mất ⇒ `ans = i − 1`; cuối cùng nếu `m == n` trả `n`. (Cài ở mục 7.3.)

In [10]:
# Minh hoạ: các ô bên trong nhóm giữ giá trị cũ nhưng hai đầu luôn đúng
n = 5
length = [0] * (n + 2)
for x in [3, 5, 1, 2, 4]:
    left, right = length[x - 1], length[x + 1]
    total = left + right + 1
    length[x - left] = length[x + right] = total
    print(f"bật {x}: left={left}, right={right}, total={total}, length[1..n] = {length[1:n+1]}")

bật 3: left=0, right=0, total=1, length[1..n] = [0, 0, 1, 0, 0]
bật 5: left=0, right=0, total=1, length[1..n] = [0, 0, 1, 0, 1]
bật 1: left=0, right=0, total=1, length[1..n] = [1, 0, 1, 0, 1]
bật 2: left=1, right=1, total=3, length[1..n] = [3, 0, 3, 0, 1]
bật 4: left=3, right=1, total=5, length[1..n] = [5, 0, 3, 0, 5]


## 7. Các cách tiếp theo

### 7.1 Cách 2 — Độ dài ở hai đầu + đếm số nhóm theo độ dài

Từng đoạn:
1. `length = [0] * (n + 2)`, `cnt = [0] * (n + 1)`, `ans = -1`.
2. Với bước `i`, vị trí `x`: lấy `left`, `right`, tính `total`.
3. Cập nhật `cnt` (bỏ hai nhóm cũ, thêm nhóm mới), cập nhật hai đầu mút.
4. Nếu `cnt[m] > 0` ⇒ `ans = i`.

In [11]:
def endpoints_count(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        cnt[left] -= 1           # cnt[0] có thể âm — vô hại, ta không đọc cnt[0]
        cnt[right] -= 1
        cnt[total] += 1
        length[x - left] = length[x + right] = total
        if cnt[m] > 0:
            ans = step
    return ans

for arr, m, want in examples:
    print(endpoints_count(arr, m), "mong đợi", want)

4 mong đợi 4
-1 mong đợi -1


Code nộp LeetCode cho cách 2:

In [12]:
class SolutionCount:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        n = len(arr)
        length = [0] * (n + 2)
        cnt = [0] * (n + 2)
        ans = -1
        for step, x in enumerate(arr, 1):
            left, right = length[x - 1], length[x + 1]
            total = left + right + 1
            if left:
                cnt[left] -= 1
            if right:
                cnt[right] -= 1
            cnt[total] += 1
            length[x - left] = length[x + right] = total
            if cnt[m]:
                ans = step
        return ans

print([SolutionCount().findLatestStep(a, m) for a, m, _ in examples])

[4, -1]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`.

### 7.2 Cách 3 — Union-Find có kích thước

Từng đoạn:
1. `parent`, `size` cho vị trí `1..n`; `on[p]` cho biết bit đã bật chưa; `cnt[L]` như cách 2.
2. Bật `x`: tạo nhóm `{x}` (`cnt[1] += 1`). Nếu `x − 1` đã bật thì `union(x − 1, x)`; tương tự `x + 1`.
3. Trong `union`: `cnt[size_a] −= 1`, `cnt[size_b] −= 1`, `cnt[size_a + size_b] += 1`.
4. Nếu `cnt[m] > 0` ⇒ `ans = step`.

In [13]:
class SolutionDSU:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        n = len(arr)
        parent = list(range(n + 2))
        size = [0] * (n + 2)
        cnt = [0] * (n + 2)

        def find(x: int) -> int:
            while parent[x] != x:
                parent[x] = parent[parent[x]]
                x = parent[x]
            return x

        def union(a: int, b: int) -> None:
            ra, rb = find(a), find(b)
            if size[ra] < size[rb]:
                ra, rb = rb, ra
            cnt[size[ra]] -= 1
            cnt[size[rb]] -= 1
            parent[rb] = ra
            size[ra] += size[rb]
            cnt[size[ra]] += 1

        ans = -1
        for step, x in enumerate(arr, 1):
            size[x] = 1
            cnt[1] += 1
            if size[find(x - 1)] and x - 1 >= 1:
                union(x - 1, x)
            if x + 1 <= n and size[find(x + 1)]:
                union(x, x + 1)
            if cnt[m]:
                ans = step
        return ans

print([SolutionDSU().findLatestStep(a, m) for a, m, _ in examples])

[4, -1]


**Độ phức tạp:** Time `O(n · α(n))`, Space `O(n)`. Ở đây `size[p] = 0` nghĩa là `p` chưa bật (vì `size` chỉ khác 0 sau khi bật, và gốc của một tập đã bật luôn có `size ≥ 1`).

### 7.3 Cách 4 — Kiểm tra trước khi gộp (không cần mảng `cnt`)

Nhận xét: một nhóm dài `m` chỉ **biến mất** khi bị gộp. Nếu ở bước `i` ta sắp gộp một nhóm dài `m` (tức `left == m` hoặc `right == m`) thì nhóm đó **tồn tại ở bước `i − 1`** ⇒ `ans = i − 1` (lấy bước muộn nhất). Nhóm dài `m` nào không bao giờ bị gộp thì tồn tại tới bước `n` — chỉ xảy ra khi nó là cả chuỗi, tức `m == n`.

In [14]:
class SolutionBeforeMerge:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        n = len(arr)
        if m == n:
            return n
        length = [0] * (n + 2)
        ans = -1
        for step, x in enumerate(arr, 1):
            left, right = length[x - 1], length[x + 1]
            if left == m or right == m:
                ans = step - 1            # nhóm dài m tồn tại ngay trước bước này
            total = left + right + 1
            length[x - left] = length[x + right] = total
        return ans

print([SolutionBeforeMerge().findLatestStep(a, m) for a, m, _ in examples])

[4, -1]


Vì sao `m < n` thì nhóm dài `m` nào cũng bị gộp? Cuối cùng chuỗi toàn `1` (một nhóm dài `n`), nên mọi nhóm nhỏ hơn đều phải hợp với hàng xóm ở một bước nào đó.

### 7.4 Cách 5 — Mảng thời điểm + cửa sổ trượt

Đảo vai: `day[p]` = bước mà vị trí `p` được bật. Một đoạn `[l, l + m − 1]` là **một nhóm đúng độ dài `m`** tại bước `t` ⇔
- mọi vị trí trong đoạn đã bật: `max(day[l..l+m−1]) ≤ t`,
- hai vị trí sát biên **chưa** bật: `t < min(day[l − 1], day[l + m])` (biên ngoài chuỗi coi như `n + 1`).

Vậy đoạn này cho các bước `t ∈ [maxWin, B − 1]` với `B = min(biên)`; nếu `maxWin < B` thì bước muộn nhất là `B − 1`. Đáp án = max qua mọi cửa sổ (hoặc `-1`). Max của mọi cửa sổ độ dài `m` tính bằng `deque` đơn điệu ⇒ `O(n)`.

In [15]:
class SolutionWindow:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        n = len(arr)
        day = [n + 1] * (n + 2)                 # day[0] = day[n+1] = n+1 (không bao giờ bật)
        for step, x in enumerate(arr, 1):
            day[x] = step
        ans = -1
        dq = deque()
        for r in range(1, n + 1):
            while dq and day[dq[-1]] <= day[r]:
                dq.pop()
            dq.append(r)
            l = r - m + 1
            if dq[0] < l:
                dq.popleft()
            if l >= 1:
                win_max = day[dq[0]]
                B = min(day[l - 1], day[r + 1])
                if win_max < B:
                    ans = max(ans, B - 1)
        return ans

print([SolutionWindow().findLatestStep(a, m) for a, m, _ in examples])

[4, -1]


**Độ phức tạp:** Time `O(n)`, Space `O(n)`. Cách này không mô phỏng theo thời gian mà xét **từng đoạn vị trí** — một kỹ thuật "đảo trục" hữu ích.

## 8. TRACE từng bước

Trace cách 2 trên ví dụ 2: `arr = [3,1,5,4,2]`, `m = 2`, `n = 5`.

| Bước | x | left | right | total | Đầu mút cập nhật | Nhóm hiện có (độ dài) | cnt[2] | ans |
|------|---|------|-------|-------|------------------|------------------------|--------|-----|
| 1 | 3 | 0 | 0 | 1 | length[3]=1 | {1} | 0 | -1 |
| 2 | 1 | 0 | 0 | 1 | length[1]=1 | {1,1} | 0 | -1 |
| 3 | 5 | 0 | 0 | 1 | length[5]=1 | {1,1,1} | 0 | -1 |
| 4 | 4 | 1 | 1 | 3 | length[3]=length[5]=3 | {1,3} | 0 | -1 |
| 5 | 2 | 1 | 3 | 5 | length[1]=length[5]=5 | {5} | 0 | **-1** |

In [16]:
def trace_endpoints(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = Counter()
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        if left:
            cnt[left] -= 1
        if right:
            cnt[right] -= 1
        cnt[total] += 1
        length[x - left] = length[x + right] = total
        if cnt[m] > 0:
            ans = step
        groups = sorted(L for L, c in cnt.items() for _ in range(c) if c > 0)
        print(f"bước {step}: x={x}, left={left}, right={right}, total={total}, "
              f"đầu mút [{x - left}, {x + right}], nhóm={groups}, cnt[{m}]={cnt[m]}, ans={ans}")
    return ans

trace_endpoints([3, 1, 5, 4, 2], 2)

bước 1: x=3, left=0, right=0, total=1, đầu mút [3, 3], nhóm=[1], cnt[2]=0, ans=-1
bước 2: x=1, left=0, right=0, total=1, đầu mút [1, 1], nhóm=[1, 1], cnt[2]=0, ans=-1
bước 3: x=5, left=0, right=0, total=1, đầu mút [5, 5], nhóm=[1, 1, 1], cnt[2]=0, ans=-1
bước 4: x=4, left=1, right=1, total=3, đầu mút [3, 5], nhóm=[1, 3], cnt[2]=0, ans=-1
bước 5: x=2, left=1, right=3, total=5, đầu mút [1, 5], nhóm=[5], cnt[2]=0, ans=-1


-1

In [17]:
trace_endpoints([3, 5, 1, 2, 4], 1)

bước 1: x=3, left=0, right=0, total=1, đầu mút [3, 3], nhóm=[1], cnt[1]=1, ans=1
bước 2: x=5, left=0, right=0, total=1, đầu mút [5, 5], nhóm=[1, 1], cnt[1]=2, ans=2
bước 3: x=1, left=0, right=0, total=1, đầu mút [1, 1], nhóm=[1, 1, 1], cnt[1]=3, ans=3
bước 4: x=2, left=1, right=1, total=3, đầu mút [1, 3], nhóm=[1, 3], cnt[1]=1, ans=4
bước 5: x=4, left=3, right=1, total=5, đầu mút [1, 5], nhóm=[5], cnt[1]=0, ans=4


4

Trace cách 5 (cửa sổ) trên ví dụ 1 để thấy mỗi đoạn vị trí "sống" trong khoảng bước nào:

In [18]:
def trace_window(arr, m):
    n = len(arr)
    day = [n + 1] * (n + 2)
    for step, x in enumerate(arr, 1):
        day[x] = step
    print("day[1..n] =", day[1:n + 1])
    ans = -1
    for l in range(1, n - m + 2):
        r = l + m - 1
        wmax = max(day[l:r + 1])
        B = min(day[l - 1], day[r + 1])
        alive = f"sống ở các bước {wmax}..{B - 1}" if wmax < B else "không bao giờ là nhóm đúng"
        if wmax < B:
            ans = max(ans, B - 1)
        print(f"  đoạn [{l},{r}]: max trong = {wmax}, biên = {B} -> {alive}")
    print("  đáp án", ans)
    return ans

trace_window([3, 5, 1, 2, 4], 1)

day[1..n] = [3, 4, 1, 5, 2]
  đoạn [1,1]: max trong = 3, biên = 4 -> sống ở các bước 3..3
  đoạn [2,2]: max trong = 4, biên = 1 -> không bao giờ là nhóm đúng
  đoạn [3,3]: max trong = 1, biên = 4 -> sống ở các bước 1..3
  đoạn [4,4]: max trong = 5, biên = 1 -> không bao giờ là nhóm đúng
  đoạn [5,5]: max trong = 2, biên = 5 -> sống ở các bước 2..4
  đáp án 4


4

## 9. Code nộp LeetCode

In [19]:
class Solution:
    def findLatestStep(self, arr: List[int], m: int) -> int:
        n = len(arr)
        length = [0] * (n + 2)        # độ dài nhóm, chỉ đúng tại hai đầu mút
        cnt = [0] * (n + 2)           # cnt[L] = số nhóm có độ dài L
        ans = -1
        for step, x in enumerate(arr, 1):
            left, right = length[x - 1], length[x + 1]
            total = left + right + 1
            cnt[left] -= 1            # cnt[0] bị trừ cũng vô hại (m >= 1)
            cnt[right] -= 1
            cnt[total] += 1
            length[x - left] = length[x + right] = total
            if cnt[m] > 0:
                ans = step
        return ans

for arr, m, want in examples:
    got = Solution().findLatestStep(arr, m)
    print(got, want)
    assert got == want

4 4
-1 -1


## 10. Vì sao thuật toán đúng

**Invariant (I):** sau bước `i`, với mọi nhóm `[l, r]` hiện có: `length[l] = length[r] = r − l + 1`; và với mọi `L ≥ 1`, `cnt[L]` = số nhóm độ dài `L`.

- **Ban đầu** không có nhóm, mọi `length` và `cnt[L ≥ 1]` bằng 0 ⇒ (I) đúng.
- **Bước bật `x`** (x đang là 0): nếu `x − 1` là `1` thì `x − 1` là **đầu phải** của nhóm sát trái (vì `x` là 0 nên nhóm dừng tại `x − 1`) ⇒ theo (I), `length[x − 1]` = độ dài nhóm đó; nếu `x − 1` là 0 thì `length[x − 1] = 0` (vị trí 0 chưa từng được ghi, vì ta chỉ ghi vào đầu mút của nhóm, toàn là vị trí `1`). Tương tự bên phải. Nhóm mới là `[x − left, x + right]` dài `total`; hai nhóm cũ biến mất. Ta cập nhật đúng hai đầu mút mới và `cnt` tương ứng ⇒ (I) giữ nguyên. Các nhóm khác không đổi.

Từ (I), sau bước `i`: "có nhóm dài đúng `m`" ⇔ `cnt[m] > 0`. Ta ghi `ans = i` mỗi khi điều này đúng, nên cuối cùng `ans` là bước **muộn nhất** thoả (hoặc `-1`).

**Cách 4** đúng vì nhóm dài `m` chỉ biến mất khi bị gộp (bật bit sát nó), và khi `m < n` mọi nhóm đều bị gộp trước khi kết thúc. **Cách 5** đúng vì điều kiện "đoạn `[l, r]` là một nhóm tối đại tại bước `t`" đúng bằng "mọi ô trong đoạn đã bật và hai ô biên chưa bật".

In [20]:
# Kiểm tra invariant (I) trực tiếp trong khi chạy trên hoán vị ngẫu nhiên
rng = random.Random(10)
for _ in range(200):
    n = rng.randint(1, 30)
    arr = rng.sample(range(1, n + 1), n)
    length = [0] * (n + 2); cnt = Counter(); bits = [0] * (n + 2)
    for x in arr:
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        if left: cnt[left] -= 1
        if right: cnt[right] -= 1
        cnt[total] += 1
        length[x - left] = length[x + right] = total
        bits[x] = 1
        s = "".join(map(str, bits[1:n + 1]))
        real = Counter(len(g) for g in s.split("0") if g)
        assert +cnt == real
        p = 1
        while p <= n:                    # kiểm tra hai đầu mút của từng nhóm
            if bits[p]:
                q = p
                while q + 1 <= n and bits[q + 1]:
                    q += 1
                assert length[p] == length[q] == q - p + 1
                p = q + 1
            else:
                p += 1
print("invariant (I) đúng ở mọi bước: OK")

invariant (I) đúng ở mọi bước: OK


## 11. Phân tích Time / Space

Gọi `n = len(arr)`.

| Cách | Time | Space |
|------|------|-------|
| Mô phỏng đếm lại | `O(n²)` | `O(n)` |
| Đầu mút + cnt | `O(n)` | `O(n)` |
| Union-Find | `O(n α(n))` | `O(n)` |
| Kiểm tra trước khi gộp | `O(n)` | `O(n)` |
| Mảng thời điểm + cửa sổ | `O(n)` | `O(n)` |

Không có khác biệt average/worst: mọi cách duyệt đủ `n` bước (trừ khi dừng sớm, nhưng ta cần bước *muộn nhất* nên không dừng sớm).

In [21]:
rng = random.Random(1562)
n = 10**5
arr = list(range(1, n + 1))
rng.shuffle(arr)
for name, S in [("count", SolutionCount), ("dsu", SolutionDSU), ("before_merge", SolutionBeforeMerge), ("window", SolutionWindow), ("Solution", Solution)]:
    t0 = time.perf_counter()
    res = [S().findLatestStep(arr, m) for m in (1, 7, 1000, n)]
    print(f"{name:12s}: {res} ({time.perf_counter() - t0:.3f}s)")

count       : [99795, 99766, -1, 100000] (0.095s)


dsu         : [99795, 99766, -1, 100000] (0.510s)
before_merge: [99795, 99766, -1, 100000] (0.062s)


window      : [99795, 99766, -1, 100000] (0.155s)
Solution    : [99795, 99766, -1, 100000] (0.111s)


## 12. Bộ test

In [22]:
METHODS = {
    "brute": brute_simulate,
    "endpoints_count": endpoints_count,
    "SolutionCount": lambda a, m: SolutionCount().findLatestStep(a, m),
    "SolutionDSU": lambda a, m: SolutionDSU().findLatestStep(a, m),
    "SolutionBeforeMerge": lambda a, m: SolutionBeforeMerge().findLatestStep(a, m),
    "SolutionWindow": lambda a, m: SolutionWindow().findLatestStep(a, m),
    "Solution": lambda a, m: Solution().findLatestStep(a, m),
}
for arr, m, want in examples:
    for name, f in METHODS.items():
        assert f(arr, m) == want, name
print("ví dụ đề: OK")

ví dụ đề: OK


In [23]:
# edge case
edge = [
    ([1], 1, 1),                    # n = 1
    ([2, 1], 2, 2),                 # m = n: chỉ có ở bước cuối
    ([2, 1], 1, 1),                 # bước 2 gộp thành 2
    ([1, 2, 3, 4], 1, 1),           # bật lần lượt: nhóm dài 1 chỉ có ở bước 1
    ([1, 3, 5, 2, 4], 1, 4),        # bước 4: "11101" vẫn còn "1" ở vị trí 5
    ([4, 3, 2, 1], 4, 4),
    ([1, 2, 3, 4], 3, 3),
    ([2, 4, 1, 3], 2, 3),           # bước 3: "1101"
    ([2, 4, 1, 3], 3, -1),
]
for arr, m, want in edge:
    for name, f in METHODS.items():
        assert f(arr, m) == want, (name, arr, m, f(arr, m))
print("edge case: OK")

edge case: OK


In [24]:
# ngẫu nhiên có seed: đối chiếu mô phỏng
rng = random.Random(2018)
for _ in range(2000):
    n = rng.randint(1, 25)
    arr = rng.sample(range(1, n + 1), n)
    m = rng.randint(1, n)
    ref = brute_simulate(arr, m)
    for name, f in METHODS.items():
        assert f(arr, m) == ref, (name, arr, m)
print("2000 test ngẫu nhiên (seed 2018): mọi cách khớp mô phỏng")

2000 test ngẫu nhiên (seed 2018): mọi cách khớp mô phỏng


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Chỉ ghi độ dài tại vị trí vừa bật

Ghi `length[x] = total` mà không cập nhật hai đầu mút ⇒ lần sau nhìn `x − 1`/`x + 1` đọc phải giá trị sai.

In [25]:
def wrong_only_x(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        cnt[left] -= 1; cnt[right] -= 1; cnt[total] += 1
        length[x] = total                    # SAI: phải ghi vào x - left và x + right
        if cnt[m] > 0:
            ans = step
    return ans

for arr, m in [([2, 1, 3], 2), ([2, 1, 3], 3), ([1, 3, 2, 4], 2), ([3, 5, 1, 2, 4], 1)]:
    print(arr, m, "| sai:", wrong_only_x(arr, m), "| đúng:", Solution().findLatestStep(arr, m))

[2, 1, 3] 2 | sai: 3 | đúng: 2
[2, 1, 3] 3 | sai: -1 | đúng: 3
[1, 3, 2, 4] 2 | sai: 4 | đúng: -1
[3, 5, 1, 2, 4] 1 | sai: 4 | đúng: 4


Ví dụ `[2,1,3]`: bước 2 tạo nhóm `[1,2]` nhưng chỉ ghi `length[1] = 2`, `length[2]` vẫn là `1`; bước 3 bật `3` đọc `length[2] = 1` ⇒ tưởng nhóm mới dài 2 thay vì 3.

Sửa: `length[x − left] = length[x + right] = total`.

### Lỗi 2 — Ghi `ans` khi **tạo ra** nhóm dài `m` thay vì khi nhóm đó **còn tồn tại**

Nhóm dài `m` có thể được tạo sớm và sống tới nhiều bước sau. Đáp án là bước muộn nhất nó còn tồn tại.

In [26]:
def wrong_on_create(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        length[x - left] = length[x + right] = total
        if total == m:
            ans = step                       # SAI: chỉ xét nhóm vừa tạo
    return ans

for arr, m, want in examples + [([1, 3, 5, 2, 4], 1, 4)]:
    print(arr, m, "| sai:", wrong_on_create(arr, m), "| đúng:", want)

[3, 5, 1, 2, 4] 1 | sai: 3 | đúng: 4
[3, 1, 5, 4, 2] 2 | sai: -1 | đúng: -1
[1, 3, 5, 2, 4] 1 | sai: 3 | đúng: 4


Sửa: duy trì `cnt[m]` và ghi `ans = step` mỗi bước mà `cnt[m] > 0` (hoặc dùng mẹo "trước khi gộp" của cách 4).

### Lỗi 3 — Quên giảm bộ đếm của các nhóm bị gộp

`cnt[m]` chỉ tăng, không bao giờ giảm ⇒ một khi có nhóm dài `m`, mọi bước sau đều bị tính.

In [27]:
def wrong_no_decrement(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        cnt[total] += 1                      # SAI: thiếu cnt[left] -= 1, cnt[right] -= 1
        length[x - left] = length[x + right] = total
        if cnt[m] > 0:
            ans = step
    return ans

for arr, m, want in examples:
    print(arr, m, "| sai:", wrong_no_decrement(arr, m), "| đúng:", want)

[3, 5, 1, 2, 4] 1 | sai: 5 | đúng: 4
[3, 1, 5, 4, 2] 2 | sai: -1 | đúng: -1


Sửa: giảm `cnt` của hai nhóm cũ trước khi tăng `cnt[total]`.

### Lỗi 4 — Cách 4 nhưng quên trường hợp `m == n`

Nhóm dài `n` không bao giờ bị gộp (nó là cả chuỗi) nên mẹo "kiểm tra trước khi gộp" không bao giờ thấy nó.

In [28]:
def wrong_forget_full(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        if left == m or right == m:
            ans = step - 1
        total = left + right + 1
        length[x - left] = length[x + right] = total
    return ans                               # SAI: thiếu "if m == n: return n"

for arr, m in [([1], 1), ([2, 1], 2), ([3, 1, 2], 3)]:
    print(arr, m, "| sai:", wrong_forget_full(arr, m), "| đúng:", Solution().findLatestStep(arr, m))

[1] 1 | sai: -1 | đúng: 1
[2, 1] 2 | sai: -1 | đúng: 2
[3, 1, 2] 3 | sai: -1 | đúng: 3


Sửa: `if m == n: return n` ngay đầu hàm.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Bật dần từng phần tử, hỏi về **các đoạn liên tiếp**" ⇒ **độ dài ở hai đầu mút** (merge intervals online) hoặc **Union-Find theo vị trí**.
- Cần biết "có đoạn độ dài `X` không" ⇒ thêm **bộ đếm theo độ dài**.
- Hỏi "**muộn nhất**" ⇒ hoặc ghi đè `ans` mỗi bước thoả, hoặc chạy ngược thời gian (gợi ý của đề), hoặc "đảo trục" sang mảng thời điểm.
- Lính canh `length[0] = length[n+1] = 0` loại bỏ mọi `if` biên.
- Bài tương tự: Longest Consecutive Sequence (LC 128) dùng đúng kỹ thuật hai đầu mút với `dict`.

## 15. Biến thể của bài

1. **Bước sớm nhất** có nhóm dài `m` (bài tập 4).
2. **Số bước** có nhóm dài đúng `m` (bài tập 5).
3. **Longest Consecutive Sequence (LC 128)**: thêm số vào tập, theo dõi đoạn dài nhất — hai đầu mút trong `dict`.
4. **K Empty Slots (LC 683)**: hoa nở theo thứ tự, tìm ngày có hai hoa nở cách nhau đúng `k` ô trống — dùng mảng thời điểm + cửa sổ (giống cách 5).
5. **Xoá dần** thay vì bật dần: chạy ngược thời gian là thành bài này.

In [29]:
# Biến thể 3 (LC 128) bằng kỹ thuật hai đầu mút với dict
def longest_consecutive(nums):
    length = {}
    best = 0
    for x in nums:
        if x in length:
            continue
        left, right = length.get(x - 1, 0), length.get(x + 1, 0)
        total = left + right + 1
        length[x] = total                    # đánh dấu x đã thêm
        length[x - left] = length[x + right] = total
        best = max(best, total)
    return best

print(longest_consecutive([100, 4, 200, 1, 3, 2]), longest_consecutive([0, 3, 7, 2, 5, 8, 4, 6, 0, 1]), "(LC 128: 4 và 9)")

4 9 (LC 128: 4 và 9)


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|------|---------|------|-------|--------------|
| 1. Mô phỏng | đếm lại nhóm mỗi bước | `O(n²)` | `O(n)` | kiểm chứng |
| 2. Đầu mút + cnt | gộp `O(1)`, đếm theo độ dài | `O(n)` | `O(n)` | **bản nộp** dễ hiểu |
| 3. Union-Find | tập theo vị trí, có size | `O(n α)` | `O(n)` | khi đã quen DSU / cần truy vấn nhóm của vị trí bất kỳ |
| 4. Trước khi gộp | nhóm `m` biến mất ⇒ `ans = i − 1` | `O(n)` | `O(n)` | gọn nhất, nhớ `m == n` |
| 5. Thời điểm + cửa sổ | đoạn `[l,r]` sống trong `[max, biên−1]` | `O(n)` | `O(n)` | tư duy "đảo trục", mở rộng sang LC 683 |

## 17. 5 bài tập vận dụng

**Bài 1 (đoán output):** `findLatestStep([1,2,3], 3)` và `findLatestStep([1,2,3], 1)`?

**Bài 2 (tính tay):** Với `arr = [4,2,6,1,3,5]`, `m = 2`, lập bảng `left, right, total, cnt[2]` qua từng bước và cho đáp án.

**Bài 3 (điền TODO):** Hoàn thiện thân vòng lặp của `ex3_latest(arr, m)` (cách 2).

**Bài 4 (tự viết lại):** Viết `ex4_earliest(arr, m)` trả về **bước sớm nhất** có nhóm dài đúng `m` (hoặc `-1`).

**Bài 5 (mini challenge):** Viết `ex5_count_steps(arr, m)` trả về **số bước** (trong `1..n`) mà sau bước đó tồn tại ít nhất một nhóm dài đúng `m`.

In [30]:
# Bài 3 — điền TODO (chỉ định nghĩa, KHÔNG gọi)
def ex3_latest(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        # TODO: lấy left, right, tính total
        # TODO: cập nhật cnt cho hai nhóm cũ và nhóm mới
        # TODO: ghi total vào hai đầu mút mới
        # TODO: nếu cnt[m] > 0 thì ans = step
        pass
    return ans

In [31]:
# Bài 4 & 5 — khung tự viết (chỉ định nghĩa, KHÔNG gọi)
def ex4_earliest(arr, m):
    # TODO
    raise NotImplementedError

def ex5_count_steps(arr, m):
    # TODO
    raise NotImplementedError

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Bật lần lượt 1, 2, 3 ⇒ chuỗi `100`, `110`, `111`. (2) Nhóm dài 3 chỉ có ở bước 3; nhóm dài 1 chỉ có ở bước 1. (3) `3` và `1`.

**Bài 2.** (1) Bước 1–3 bật `4, 2, 6` riêng lẻ (ba nhóm dài 1). (2) Bước 4 bật `1`: gộp với `2` ⇒ nhóm `[1,2]` dài 2, `cnt[2] = 1`. (3) Bước 5 bật `3`: gộp `[1,2]`, `3`, `[4]` ⇒ dài 4, `cnt[2] = 0`; bước 6 bật `5`: gộp thành 6. Đáp án `4`.

**Bài 3.** (1) `left, right = length[x−1], length[x+1]; total = left + right + 1`. (2) `cnt[left] −= 1; cnt[right] −= 1; cnt[total] += 1`. (3) `length[x−left] = length[x+right] = total; if cnt[m] > 0: ans = step`.

**Bài 4.** (1) Giống cách 2 nhưng `return step` ngay lần đầu `cnt[m] > 0`. (2) Hoặc: lần đầu `total == m` (nhóm dài `m` đầu tiên phải là nhóm vừa được tạo). (3) Không có thì `-1`.

**Bài 5.** (1) Giữ cách 2. (2) Mỗi bước `cnt[m] > 0` thì tăng biến đếm. (3) Kiểm chứng bằng mô phỏng `O(n²)`.

## 19. Đáp án tham khảo cho bài tập

In [32]:
# Bài 1
a1 = (Solution().findLatestStep([1, 2, 3], 3), Solution().findLatestStep([1, 2, 3], 1))
print("Bài 1:", a1)
assert a1 == (3, 1)

# Bài 2
a2 = trace_endpoints([4, 2, 6, 1, 3, 5], 2)
assert a2 == 4 == brute_simulate([4, 2, 6, 1, 3, 5], 2)

Bài 1: (3, 1)
bước 1: x=4, left=0, right=0, total=1, đầu mút [4, 4], nhóm=[1], cnt[2]=0, ans=-1
bước 2: x=2, left=0, right=0, total=1, đầu mút [2, 2], nhóm=[1, 1], cnt[2]=0, ans=-1
bước 3: x=6, left=0, right=0, total=1, đầu mút [6, 6], nhóm=[1, 1, 1], cnt[2]=0, ans=-1
bước 4: x=1, left=0, right=1, total=2, đầu mút [1, 2], nhóm=[1, 1, 2], cnt[2]=1, ans=4
bước 5: x=3, left=2, right=1, total=4, đầu mút [1, 4], nhóm=[1, 4], cnt[2]=0, ans=4
bước 6: x=5, left=4, right=1, total=6, đầu mút [1, 6], nhóm=[6], cnt[2]=0, ans=4


In [33]:
# Bài 3
def ex3_latest_sol(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    ans = -1
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        cnt[left] -= 1
        cnt[right] -= 1
        cnt[total] += 1
        length[x - left] = length[x + right] = total
        if cnt[m] > 0:
            ans = step
    return ans

rng3 = random.Random(3)
for _ in range(500):
    n = rng3.randint(1, 20)
    arr = rng3.sample(range(1, n + 1), n)
    m = rng3.randint(1, n)
    assert ex3_latest_sol(arr, m) == brute_simulate(arr, m)
print("Bài 3: OK")

Bài 3: OK


In [34]:
# Bài 4
def ex4_earliest_sol(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    for step, x in enumerate(arr, 1):
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        length[x - left] = length[x + right] = total
        if total == m:
            return step
    return -1

def ex4_brute(arr, m):
    n = len(arr)
    bits = ["0"] * n
    for step, x in enumerate(arr, 1):
        bits[x - 1] = "1"
        if any(len(g) == m for g in "".join(bits).split("0")):
            return step
    return -1

rng4 = random.Random(4)
for _ in range(800):
    n = rng4.randint(1, 20)
    arr = rng4.sample(range(1, n + 1), n)
    m = rng4.randint(1, n)
    assert ex4_earliest_sol(arr, m) == ex4_brute(arr, m)
print("Bài 4: OK, ví dụ:", ex4_earliest_sol([3, 5, 1, 2, 4], 1), ex4_earliest_sol([3, 5, 1, 2, 4], 3))

Bài 4: OK, ví dụ: 1 4


In [35]:
# Bài 5
def ex5_count_steps_sol(arr, m):
    n = len(arr)
    length = [0] * (n + 2)
    cnt = [0] * (n + 2)
    steps = 0
    for x in arr:
        left, right = length[x - 1], length[x + 1]
        total = left + right + 1
        cnt[left] -= 1
        cnt[right] -= 1
        cnt[total] += 1
        length[x - left] = length[x + right] = total
        if cnt[m] > 0:
            steps += 1
    return steps

def ex5_brute(arr, m):
    n = len(arr)
    bits = ["0"] * n
    steps = 0
    for x in arr:
        bits[x - 1] = "1"
        if any(len(g) == m for g in "".join(bits).split("0")):
            steps += 1
    return steps

rng5 = random.Random(5)
for _ in range(800):
    n = rng5.randint(1, 20)
    arr = rng5.sample(range(1, n + 1), n)
    m = rng5.randint(1, n)
    assert ex5_count_steps_sol(arr, m) == ex5_brute(arr, m)
print("Bài 5: OK, ví dụ:", ex5_count_steps_sol([3, 5, 1, 2, 4], 1), ex5_count_steps_sol([3, 1, 5, 4, 2], 2))

Bài 5: OK, ví dụ: 4 0


## 20. Template nhớ nhanh + Checklist

```python
class Solution:
    def findLatestStep(self, arr, m):
        n = len(arr)
        length = [0] * (n + 2)
        cnt = [0] * (n + 2)
        ans = -1
        for step, x in enumerate(arr, 1):
            left, right = length[x - 1], length[x + 1]
            total = left + right + 1
            cnt[left] -= 1; cnt[right] -= 1; cnt[total] += 1
            length[x - left] = length[x + right] = total
            if cnt[m] > 0:
                ans = step
        return ans
```

**Template "gộp đoạn online":** `length[đầu trái] = length[đầu phải] = độ dài`; bật `x` ⇒ `left = length[x−1]`, `right = length[x+1]`, đoạn mới `[x−left, x+right]`.

**Checklist trước khi submit:**
- [ ] Mảng `length` dài `n + 2` (lính canh hai đầu).
- [ ] Ghi độ dài vào **hai đầu mút mới**, không phải vào `x`.
- [ ] Giảm `cnt` của **cả hai** nhóm bị gộp.
- [ ] Bước đánh số từ **1** (`enumerate(arr, 1)`).
- [ ] Nếu dùng mẹo "trước khi gộp": xử lý `m == n`.
- [ ] Thử nhanh: ví dụ 1 → `4`, ví dụ 2 → `-1`, `([1],1) → 1`, `([2,1],2) → 2`.